## Module 6-3 Forecasting Continuous Earnings with XGBoost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb

### 1. Recap: load, filter, and engineer features

Nothing here is new.

In [ ]:
df = pd.read_csv('../data/comp_sample.csv', 
                 dtype={"gvkey": str,})

df = df[
    (df['indfmt'] == 'INDL') &
    (df['curcd'] == 'USD') &
    (df['costat'] == 'A')
].drop_duplicates(subset = ['gvkey', 'fyear']).copy()
df = df.sort_values(['gvkey', 'fyear']).reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
ID_COLS = ['gvkey', 'datadate', 'fyear']
NON_FS_COLS = ['au']  # e.g. auditor code: an identifier, not a financial-statement item

# Items that should NOT be scaled by total assets:
NO_SCALE_COLS = ['at', 'csho', 'prcc_f']

exclude_cols = set(ID_COLS + NON_FS_COLS)
numeric_cols = df.select_dtypes(include='number').columns
predictor_base_cols = [c for c in numeric_cols if c not in exclude_cols]

print(f'{len(predictor_base_cols)} raw financial-statement columns auto-detected as predictors:')
print(predictor_base_cols)

In [ ]:
def build_features(data: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Current value, lagged value, and percentage change for each base column."""
    at_cur = data['at']
    at_lag = data.groupby('gvkey')['at'].shift(1).where(data.groupby('gvkey')['fyear'].diff()==1)

    feature_cols = {}
    for col in base_cols:
        cur = data[col]
        lag = data.groupby('gvkey')[col].shift(1).where(data.groupby('gvkey')['fyear'].diff()==1)

        if col in NO_SCALE_COLS:
            cur_scaled, lag_scaled = cur, lag
        else:
            cur_scaled = (cur / at_cur).replace([np.inf, -np.inf], np.nan)
            lag_scaled = (lag / at_lag).replace([np.inf, -np.inf], np.nan)

        pct_change = ((cur - lag) / lag.abs()).replace([np.inf, -np.inf], np.nan)

        feature_cols[f'{col}_cur'] = cur_scaled
        feature_cols[f'{col}_lag'] = lag_scaled
        feature_cols[f'{col}_pctchg'] = pct_change

    return pd.DataFrame(feature_cols, index=data.index)


X_all = build_features(df, predictor_base_cols)
feature_cols = X_all.columns.tolist()
print(X_all.shape)
X_all.head()

### 2. Build a continuous earnings target

In addition, we need to **winsorize the target**: Raw EPS has a few extreme outliers. We winsorize at the 1st/99th percentile of the **training period only** — never the validation or test period.

In [ ]:
df['eps'] = (df['ni'] / df['csho']).replace([np.inf, -np.inf], np.nan)
df['eps_lead'] = df.groupby('gvkey')['eps'].shift(-1).where(df.groupby('gvkey')['fyear'].diff(-1)==-1)

model_df = pd.concat(
    [df[['gvkey', 'fyear', 'eps', 'eps_lead', 'prcc_f']], X_all], axis=1
)
model_df = model_df.dropna(subset=['eps', 'eps_lead', 'prcc_f']).reset_index(drop=True)
model_df = model_df[model_df['prcc_f'] >= 1].reset_index(drop=True)

print(model_df.shape)
model_df[['gvkey', 'fyear', 'eps', 'eps_lead', 'prcc_f']].head(10)

### 3. Split chronologically, then winsorize using training bounds only

In [ ]:
train_mask = model_df['fyear'] <= 2019
lo, hi = model_df.loc[train_mask, 'eps_lead'].quantile([0.01, 0.99])

model_df['eps_lead'] = model_df['eps_lead'].clip(lo, hi)
model_df['eps'] = model_df['eps'].clip(lo, hi)

train = model_df[model_df['fyear'] <= 2019]
val = model_df[(model_df['fyear'] > 2019) & (model_df['fyear'] <= 2021)]
test = model_df[model_df['fyear'] == 2022]

X_train, y_train = train[feature_cols], train['eps_lead']
X_val, y_val = val[feature_cols], val['eps_lead']
X_test, y_test = test[feature_cols], test['eps_lead']

price_test = test['prcc_f']
print(f'train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')

### 4. XGBoost

In [ ]:
xgb_tuned = xgb.XGBRegressor(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:pseudohubererror',
    eval_metric='mae',
    early_stopping_rounds=50,
    random_state=42,
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f'Stopped after {xgb_tuned.best_iteration + 1} trees '
      f'(validation MAE = {xgb_tuned.best_score:.3f})')

tuned_pred = xgb_tuned.predict(X_test)

### 7. Evaluate: MAFE, RMSE, and MDAFE, scaled by price

Following Chattopadhyay et al. (2025, Section 3.3), we evaluate the model with three complementary metrics, each computed on the **absolute forecast error scaled by price**:

- **MAFE** (mean absolute forecast error): average error magnitude — sensitive to every observation equally.
- **RMSE** (root mean squared error): penalizes large misses more heavily than MAFE.
- **MDAFE** (median absolute forecast error): robust to outliers, describing the "typical" firm's forecast error.

We also winsorize the scaled *errors* themselves at the 1st and 99th percentiles before aggregating to minimize the effect of outliers.

In [ ]:
def forecast_metrics(y_true, y_pred, price) -> dict[str, float]:
    scaled_error = (np.asarray(y_pred) - np.asarray(y_true)) / np.asarray(price)
    abs_error = pd.Series(np.abs(scaled_error))
    signed_error = pd.Series(scaled_error)
    abs_error_w = abs_error.clip(*abs_error.quantile([0.01, 0.99]))
    signed_error_w = signed_error.clip(*signed_error.quantile([0.01, 0.99]))
    return {
        'MAFE': abs_error_w.mean(),
        'RMSE': np.sqrt((signed_error_w ** 2).mean()),
        'MDAFE': abs_error_w.median(),
    }
print(forecast_metrics(y_test, tuned_pred, price_test))

### 6. Which predictors matter most? (Optional)

In [ ]:
importances = pd.Series(
    xgb_tuned.feature_importances_, index=feature_cols
).sort_values(ascending=False)

top_n = 20
plt.figure(figsize=(8, 6))
importances.head(top_n).sort_values().plot(kind='barh')
plt.xlabel('XGBoost feature importance (gain)')
plt.title(f'Top {top_n} predictors of next-year EPS')
plt.tight_layout()
plt.show()